# Silver — CRM Sales Details
Sales transactions from the CRM.

`bronze.crm_sales_details` → `silver.crm_sales`

## Init

In [ ]:
import os, sys
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DateType

# Make src/ importable from wherever this notebook runs (git folder, bundle, VS Code sync)
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src")) and root != "/":
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from lakehouse.transforms import trim_strings, normalize, rename, yyyymmdd_to_date

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.crm_sales_details")

## Transformations

### Trim all string columns

In [ ]:
df = trim_strings(df)

### Clean dates
Dates arrive as integers like `20101229`. Zero or wrong-length values are invalid → `NULL`.

In [ ]:
df = yyyymmdd_to_date(df, "sls_order_dt", "sls_ship_dt", "sls_due_dt")

### Fix invalid prices
If price is missing or non-positive, derive it from `sales / quantity`.

In [ ]:
df = df.withColumn(
    "sls_price",
    F.when(
        col("sls_price").isNull() | (col("sls_price") <= 0),
        F.when(col("sls_quantity") != 0, col("sls_sales") / col("sls_quantity")).otherwise(None)
    ).otherwise(col("sls_price"))
)

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
df = rename(df, RENAME_MAP)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.silver.crm_sales")

In [ ]:
%sql
SELECT * FROM workspace.silver.crm_sales LIMIT 10;